In [ ]:
# Init environment before running a demo notebook.
from resources.utils import *  
from resources.dask_utils import *
import pprint

init_demo()
init_dask_cluster_staging(scale=2)

# Reload the global vars again
from resources.utils import *  
from resources.dask_utils import *  

pp = pprint.PrettyPrinter(indent=2, width=80, sort_dicts=False, compact=True)

# You can check here the number of workers, threads and memory per worker.
# In local mode, you can configure them by running e.g.
# DASK_MEMORY_EOPF=4G DASK_CORES_EOPF=4 docker compose up # ...
display(dask_cluster_staging)

In [ ]:
# Create a test collection
CATALOG_COLLECTION_ID = "SPRINT_34_RSPY_618_TEST_COLLECTION"
collection = create_test_collection(CATALOG_COLLECTION_ID)
items = catalog_client.get_items(CATALOG_COLLECTION_ID)
list(items)

In [ ]:
cadip_no_geometry_no_bbox = cadip_client.search(
    method="GET", 
    stac_filter="externalIds=cadip:6f3c8d91-2b0e-492d-aef6-87b24f2bcb1e")
cadip_no_geometry_no_bbox

In [ ]:
# 1. 
# Insert a CADIP session STAC item without geometry nor bbox. Check that the item is successfully inserted.
fc = cadip_no_geometry_no_bbox.to_dict()
item_dict = fc["features"][0]
item_dict["collection"] = CATALOG_COLLECTION_ID
item_dict.setdefault("stac_extensions", [])
item_dict["assets"] = {}  # Avoid triggering the bucket-transfer logic (the temp S3 copy/move of assets)

owner = catalog_client.owner_id
url = f"{catalog_client.href_service}/catalog/collections/{owner}:{CATALOG_COLLECTION_ID}/items"

if cluster_mode:
    resp = http_session.post(url, json=item_dict, headers={"x-api-key": os.environ["RSPY_APIKEY"]}, timeout=120)
else:
    resp = http_session.post(url, json=item_dict, timeout=120)

print(resp.status_code)
print(resp.text)


In [ ]:
import copy, pystac

In [ ]:
result = list(catalog_client.get_collection(CATALOG_COLLECTION_ID).get_items())
result

In [ ]:
item_collection_prip = prip_client.search(
    method='GET',
    stac_filter="externalIds=prip:4db05e5e-16d7-4c15-8ca1-9a7d31d06eba, b99c8f80-ee84-4854-bd36-15b18ac0ecca")
item_collection_prip

In [ ]:
# 2.
# Insert an item with an invalid geojson geometry. 
# Check that HTTP 400 bad request error is returned with a clear error message and that the item is NOT added to the catalog.
item = copy.deepcopy(item_collection_prip.to_dict()['features'][0])
item["collection"] = CATALOG_COLLECTION_ID
# invalid geojson geometry (not an object)
item["geometry"] = "not-an-object"
item.pop("bbox", None)  # 

owner = catalog_client.owner_id
url = f"{catalog_client.href_service}/catalog/collections/{owner}:{CATALOG_COLLECTION_ID}/items"
r = http_session.post(url, json=item, timeout=120)
print(r.status_code)
print(r.text)  # error message

In [ ]:
# 2.
# Insert an item with an invalid geojson geometry. 
# Check that HTTP 400 bad request error is returned with a clear error message and that the item is NOT added to the catalog.
item = copy.deepcopy(item_collection_prip.to_dict()['features'][0])
item["collection"] = CATALOG_COLLECTION_ID
# invalid geojson geometry (ring not closed)
item["geometry"] = {"type": "Polygon", "coordinates": [[[0,0],[1,0],[1,1],[0,1]]]}
item.pop("bbox", None)  #

owner = catalog_client.owner_id
url = f"{catalog_client.href_service}/catalog/collections/{owner}:{CATALOG_COLLECTION_ID}/items"
r = http_session.post(url, json=item, timeout=120)
print(r.status_code)
print(r.text)  # error message

In [ ]:
# 3.
# Insert an item with a valid geojson geometry but no bbox.
# Check that RS-Server computes and adds the bbox as per STAC-CORE-ITEM-REQ-0230 requirement.
item = copy.deepcopy(item_collection_prip.to_dict()['features'][0])
item["collection"] = CATALOG_COLLECTION_ID
# invalid geojson geometry (nu e obiect)
item.pop("bbox", None)  # no bbox
item["assets"] = {}  # Avoid triggering the bucket-transfer logic (the temp S3 copy/move of assets)

owner = catalog_client.owner_id
url = f"{catalog_client.href_service}/catalog/collections/{owner}:{CATALOG_COLLECTION_ID}/items"
r = http_session.post(url, json=item, timeout=120)
print(r.status_code)
print(r.text)  # 

In [ ]:
# 4.
# Try to replace the contents of a valid item with an invalid geojson geometry using PUT. 
# Check that HTTP 400 bad request error is returned with a clear error message and that the item is NOT modified in the catalog.

import copy, pystac

COLL = CATALOG_COLLECTION_ID
owner = catalog_client.owner_id

# 1) 
item_id = "S2B_OPER_MSI_L0__GR_2BPS_20250801T074015_S20250801T070620_D04_N05.11"  # id
original = catalog_client.get_item(COLL, item_id).to_dict()

# 2) (invalid GeoJSON)
bad = copy.deepcopy(original)
# bad["geometry"] = {"type": "Polygon", "coordinates": []}  # invalid
ring = original["geometry"]["coordinates"][0]
rev = ring[:-1][::-1]
bad["geometry"] = {"type": "Polygon", "coordinates": [rev + [rev[0]]]}


# 3) PUT (replace)
url = f"{catalog_client.href_service}/catalog/collections/{owner}:{COLL}/items/{item_id}"
resp = http_session.put(url, json=bad, timeout=120)

print(resp.status_code)
print(resp.text)  # 400

# 4) not modified in catalog
after = catalog_client.get_item(COLL, item_id).to_dict()
assert after["geometry"] == original["geometry"]
assert after.get("bbox") == original.get("bbox")


In [ ]:
# 5. Same as above with PATCH.
# Try to replace the contents of a valid item with an invalid geojson geometry using PATCH. 
# Check that HTTP 400 bad request error is returned with a clear error message and that the item is NOT modified in the catalog.

owner = catalog_client.owner_id
url = f"{catalog_client.href_service}/catalog/collections/{owner}:{COLL}/items/{item_id}"

payload = {"geometry": {"type": "Polygon", "coordinates": []}, "properties": {}}
r = http_session.patch(url, json=payload, timeout=120)

print(r.status_code)
print(r.text)

In [ ]:
# 6. 
# Try to remove bbox of a valid item using PUT. 
# Check that RS-Server computes again the bbox and adds it back, so that the item in catalog still has a bbox.
import copy

owner = catalog_client.owner_id
url = f"{catalog_client.href_service}/catalog/collections/{owner}:{COLL}/items/{item_id}"

original = catalog_client.get_item(COLL, item_id).to_dict()

# PUT - bbox removed
modified = copy.deepcopy(original)
modified.pop("bbox", None)

r = http_session.put(url, json=modified, timeout=120)
print(r.status_code, r.text)

# bbox was re-computed
after = catalog_client.get_item(COLL, item_id).to_dict()
assert after["bbox"] is not None
print(after["bbox"])


In [ ]:
# Try to remove bbox of a valid item using PATCH.
# Check that RS-Server computes again the bbox and adds it back, so that the item in catalog still has a bbox.
owner = catalog_client.owner_id
url = f"{catalog_client.href_service}/catalog/collections/{owner}:{COLL}/items/{item_id}"

# PATCH: remove bbox (middleware recomputes from geometry)
payload = {"bbox": None, "properties": {}}
r = http_session.patch(url, json=payload, timeout=120)
print(r.status_code, r.text)

after = catalog_client.get_item(COLL, item_id).to_dict()
assert after["bbox"] is not None
print(after["bbox"])


In [ ]:
import copy, pystac

COLL = CATALOG_COLLECTION_ID
item_id = "S1A_20210410031928012345"

# pystac.Item
item = catalog_client.get_item(COLL, item_id)

# clone + bad geometry
bad_item = pystac.Item.from_dict(item.to_dict())
bad = bad_item.to_dict()
bad["geometry"] = {"type": "Polygon", "coordinates": "not-an-array"}  # invalid
bad_item = pystac.Item.from_dict(bad)

# PUT via client (400)
try:
    catalog_client.update_item(bad_item)
except Exception as e:
    print("update_item failed:", e)


In [ ]:
result = list(catalog_client.get_collection(CATALOG_COLLECTION_ID).get_items())
result

In [ ]:
result = catalog_client.remove_collection(CATALOG_COLLECTION_ID)
assert result.json()["deleted collection"] == CATALOG_COLLECTION_ID
pp.pprint(result.json())